<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

<!-- JT: Summary recommended TODOs:

- We shouldn't try to sell u_t = K_d d_t + K_{\phi} \phi_t as a derivation of the previously introduced PID equation. We should just frame it as "we learned about PID in the PID LX. The Duckiebot is controlled through a from a different state-feedback approach called pole-placement approach, and the final equation resembles a PID one. 
(and maybe proceed to outline the pole-placement proof so at the least all the variables that appear in the code are properly introduced and justified.) 
-->

# Lane Control

## Constant velocity PID control

With the state estimation provided by the Histogram filter, a PID controller is used to compute the commands to be sent to the Duckiebot wheels. For more details on PID control, review the dedicated [Duckietown (ente) PID control LX](https://docs.duckietown.com/ente/duckietown-manual/60-learning-experiences/lx-setup-pid-control.html).

The PID controller calculates the instantaneous control signal $u_t = [\bar v, \omega_t]^T$ based on the error $e_t = x_{ref} - \hat x_t$ between the desired reference state ($x_{ref} = [d_{ref}, \phi_{ref}]^T$) and the estimated state ($\hat x_{t} = [\hat d_t , \hat \phi_t]^T$):

$$
u_t = K_p e_t + K_i \int_0^t e_{\tau} d \tau + K_d \frac{d e_t}{d_t},
$$

<!-- JT: Try getting from PID control above to weird bi-proportional control below... (spoiler: you won't get there)
$$ \begin{aligned}
[\bar v, \omega_t]^T &= K_p [d_{ref} - \hat d_t, \phi_{ref} - \hat \phi_t]^T + K_i \int_0^t [d_{ref} - \hat d_{\tau}, \phi_{ref} - \hat \phi_{\tau}]^T d \tau + K_d \frac{d [[d_{ref} - \hat d_{\tau}, \phi_{ref} - \hat \phi_{\tau}]^T]}{d_t} =  \\

&= -K_p [\hat d_t, \hat \phi_t]^T - K_i \int_0^t [\hat d_{\tau}, \hat \phi_{\tau}]^T d \tau - K_d \frac{d [[\hat d_{\tau},\hat \phi_{\tau}]^T]}{d_t} = \dots ?
\end{aligned}
$$
-->

where:

* $K_p$ is the proportional gain, weighting the current error.
* $K_i$ is the integral gain, correcting cumulative past errors.
* $K_d$ is the derivative gain, anticipating future errors based on the rate of change.

In the context of lane following, the reference state is $[d_{ref}, \phi_{ref}]^T = [0,0]^T$, i.e., the desired outcome is for the Duckiebot to always be at the center of the lane, looking forward.  

The output of the controller is a control input $u_t = [\bar v, \omega_t]^T$, where the Duckiebot's linear velocity $\bar v$ is assumed contant for simplicity, and a variable angular velocity $\omega_t$ is computed to stay on track.

Using a linearized kinematic model and a constant velocity $\bar v$, the control law simplifies to:

<!--
JT: abuse of notation alert. The K_d below has nothing to do with the K_d above. 
-->

$$
u_t = K_d d_t + K_{\phi} \phi_t,
$$

where $K_d$ and $K_{\phi}$ are tuned proportional gains for lateral and angular errors, respectively.

<!--
JT: statement above is not true. The control law below (u_t = K_d d_t + K_{\phi} \phi_t) is obtained through:

- linearized kinematics
- small angles approximation
- pole placement approach
- other very weird assumptions. 

It does not come from the PID control equation introduced above.

- See [proof here - slides 28-31](https://docs.google.com/presentation/d/15awvCABgWQhjMkF6bMfAIevz-468u92V/edit?usp=sharing&ouid=103383252109471807908&rtpof=true&sd=true). 


2. Why do we want to do this, anyways? PID is more intuitive and should be properly implemented. 
-->

## Tuning the PID gains

Tuning the PID gains is one of the most important aspects for the stability and performance of your Duckiebot, as:

* The proportional gain ($K_p$) affects the magnitude of corrections. Too high a value leads to oscillations, while too low a value results in sluggish response.
* The integral gain ($K_i$) addresses steady-state errors but can introduce instability if over tuned.
* The derivative gain ($K_d$) smooths out the response by reducing overshoot but can amplify noise.

These are located in the config for the [lane controller node](../packages/dt-core/packages/lane_control/config/lane_controller_node/). Here are the defaults:

<!--
JT @liampaull, note the comments below and remove/update as needed
-->

```yaml 
v_bar: 0.19 #constant forward velocity (if gain k=1 in odometry calibration) 
k_d: -2.0 # as above 
k_theta: -3.0 # as above 
k_Id: -3.0 # Not introduced above
k_Iphi: 0.0 # Not introduced above
theta_thres_max: 0.75 # Part of the proof linked above, nothing to do with PID
theta_thres_min: -0.5 # Part of the proof linked above, nothing to do with PID
d_thres: 0.2615 # Weird hack from the proof linked above, nothing to do with PID
d_offset: 0.0  # Weird hack from the proof linked above, nothing to do with PID

integral_bounds: # JT: where is the integral in u_t = K_d d_t + K_{\phi} \phi_t?
  d:
    top: 0.3
    bot: -0.3
  phi:
    top: 1.2
    bot: -1.2
```

The are defined as follows: 

- $\bar v$:  Nominal forward velocity (m/s). This is the cruising speed used when no stop line is detected; velocity is reduced as the robot approaches a stop line.

<!--
JT: 
(i) there is no stop line in lane following, and mentioning it is just confusing imo
(ii) we should remark that the "gain" coefficient tuned during the odometry calibration procedure along with the trim is, de facto, the nominal way to change the v_bar. Arguably v_bar should be removed. 
-->

- $k_d$: Proportional gain on lateral error $d$. Negative because a positive lateral deviation (too far left) should produce a negative angular velocity (turn right). Larger magnitude = more aggressive lateral correction.

<!--
JT: 
(i) it has previously been introduced as K_d, not k_d
(ii) very abusive notation as it is confused with K_d of the PID controller intro (which, at this point, is misleading)
(iii) I call BS on the sign explanation. I bet it's negative because this controller is a hack.
-->

- $k_theta$:  Proportional gain on heading error φ. Negative for the same sign convention reason. Typically larger than k_d since heading error is easier to correct quickly.

<!--
JT: 
- there is no such thing as k_theta, it's been previously introduced as K_{\phi}
- I call BS on the sign explanation. I bet it's negative because this controller is a hack.
-->

- k_Id:  Integral gain on lateral error. Accumulates d error over time to eliminate steady-state lateral offset (e.g. a consistent bias from road camber or calibration error).

<!--
JT: 
- there is no such thing as k_Id in the explanation above
- what is "camber"?
-->

- k_Iphi: Integral gain on heading error. Currently disabled. Would eliminate steady-state heading bias but can cause oscillation, hence left at zero.

<!--
JT: 
- there is no such thing as k_Iphi in the explanation above
- this thing should not exist, it's a hack
-->

- theta_thres_max / theta_thres_min: Asymmetric clamp on φ error before it enters the control law. Large heading errors (e.g. at intersections) are saturated to prevent extreme angular velocity commands. Asymmetric values allow different sensitivity for left vs. right heading deviations.

<!--
JT: 
- what is "φ" - we're using latex: \phi, \theta, \whatever...
- this thing should not exist, it's a hack. There are other way to limit the angular velocity (refer to robot kinematics calibration notebook. There is an "omega max" variable that at most should be picked up here.)
-->

- d_thres:  Clamp on lateral error d before it enters the control law (~half a lane width). Prevents the controller from commanding extreme corrections when the robot is far off-center.

<!--
JT: 
- never heard the word "clamp" in this context. We mean a bound? A saturation perhaps?
- it we want to avoid extreme comman
-->

- d_offset: Shifts the target lateral position away from lane center. Positive values make the robot drive to the left of center, negative to the right. Useful for avoiding obstacles or adjusting lane position.

<!--
JT: 
* target -> reference
-->

- integral_bounds.d: Anti-windup limits for the lateral integral term. The accumulated $d$ error is clamped to this range to prevent integrator windup when the robot is held off-center for a long time.

- integral_bounds.phi: Anti-windup limits for the heading integral term (disabled if k_Iphi = 0).

<!--
JT: 
* meh
-->


For more details about PID control please refer to the [Control Learning Experience](https://docs.duckietown.com/ente/duckietown-manual/60-learning-experiences/lx-setup-pid-control.html).

The velocity and steering values are turned into actuator values using inverse kinematics by [the kinematics node](../packages/dt-core/packages/robots/duckiebot/dagu_car/src/kinematics_node.py). For more details about direct and inverse kinematics refer to the [Kinematics and Odometry Learning Experience](https://docs.duckietown.com/ente/duckietown-manual/60-learning-experiences/lx-setup-modeling-kinematics.html).

We now have understood the entire autonomy stack, from the data that comes in through the sensors (camera and encoders) to the actuator commands that are sent to the wheels. One last piece to discuss is how we build a "finite state machine" that sits on top of this stack and manages what the macro-level behaviour of the robot should be. 

In this relatively simple autonomous behavior case we have two (autonomy) states: `LANE_FOLLOWING`, and `NORMAL_JOYSTICK_CONTROL` that can  be toggled using the `keyboard_control` GUI. For details, proceed to the [the next notebook about the finite state machine](./05_finite_state_machine.ipynb).